# Выбор локации для нефтяной скважины

**Описание**

В добывающей компании нужно решить, где бурить новую скважину.
Вам предоставлены пробы нефти в трёх регионах. Характеристики для каждой скважины в регионе уже известны.

> Шаги для выбора локации (обычно):
> 1. В избранном регионе собирают характеристики для скважин: качество нефти и объём её запасов;
> 1. Строят модель для предсказания объёма запасов в новых скважинах;
> 1. Выбирают скважины с самыми высокими оценками значений;
> 1. Определяют регион с максимальной суммарной прибылью отобранных скважин.


**Цель**

Построить модель для определения региона, где добыча принесёт наибольшую прибыль.

**Описание данных**

Данные геологоразведки трёх регионов находятся в файлах:

* `https://code.s3.yandex.net/datasets/geo_data_0.csv`
* `https://code.s3.yandex.net/datasets/geo_data_1.csv`
* `https://code.s3.yandex.net/datasets/geo_data_2.csv`
* `id` — уникальный идентификатор скважины;
* `f0`, `f1`, `f2` — три признака точек (неважно, что они означают, но сами признаки значимы);
* `product` — объём запасов в скважине (тыс. баррелей).

**Условия задачи:**

* Для обучения модели подходит только линейная регрессия (остальные — недостаточно предсказуемые).
* При разведке региона исследуют 500 точек, из которых с помощью машинного обучения выбирают 200 лучших для разработки.
* Бюджет на разработку скважин в регионе — 10 млрд рублей.
* При нынешних ценах один баррель сырья приносит 450 рублей дохода. Доход с каждой единицы продукта составляет 450 тыс. рублей, поскольку объём указан в тысячах баррелей.
* После оценки рисков нужно оставить лишь те регионы, в которых вероятность убытков меньше 2.5%. Среди них выбирают регион с наибольшей средней прибылью.


**План работы**
1. Изучить общую информацию о данных
1. Выполнить предобработку входных данных
1. Обучить модели
1. Выполнить подготовку к расчёту прибыли
1. Выполнить расчёт прибыли и рисков
1. Подготовить общий вывод

## Загрузка и подготовка данных

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
region_0 = pd.read_csv('https://code.s3.yandex.net/datasets/geo_data_0.csv')
region_0.head()

,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


In [3]:
region_0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB


In [4]:
region_0.describe()

,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.500419,0.250143,2.502647,92.500000
std,0.871832,0.504433,3.248248,44.288691
min,-1.408605,-0.848218,-12.088328,0.000000
25%,-0.072580,-0.200881,0.287748,56.497507
50%,0.502360,0.250252,2.515969,91.849972
75%,1.073581,0.700646,4.715088,128.564089
max,2.362331,1.343769,16.003790,185.364347


In [5]:
region_0.duplicated().sum()

np.int64(0)

In [6]:
region_1 = pd.read_csv('https://code.s3.yandex.net/datasets/geo_data_1.csv')
print(f'Размерность данных: {region_1.shape}\nЧисло дубликатов: {region_1.duplicated().sum()}\n')
print(region_1.info())
print(region_1.describe())

Размерность данных: (100000, 5)
Число дубликатов: 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        1.141296      -4.796579       2.494541      68.825000
std         8.965932       5.119872       1.703572      45.944423
min       -31.609576     -26.358598      -0.018144       0.000000
25%        -6.298551      -8.267985       1.000021      26.953261
50%         1.153055      -4.813172       2.011479      57.085625
75%         8.621015      -1.332816       3.99

In [7]:
region_2 = pd.read_csv('https://code.s3.yandex.net/datasets/geo_data_2.csv')
print(f'Размерность данных: {region_2.shape}\nЧисло дубликатов: {region_2.duplicated().sum()}\n')
print(region_2.info())
print(region_2.describe())

Размерность данных: (100000, 5)
Число дубликатов: 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.002023      -0.002081       2.495128      95.000000
std         1.732045       1.730417       3.473445      44.749921
min        -8.760004      -7.084020     -11.970335       0.000000
25%        -1.162288      -1.174820       0.130359      59.450441
50%         0.009424      -0.009482       2.484236      94.925613
75%         1.158535       1.163678       4.85

## Обучение и проверка модели

In [8]:
def train_and_evaluate(data):
    X_train, X_test, y_train, y_test = train_test_split(
        data[['f0', 'f1', 'f2']], data['product'], test_size=0.2,
        random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return {'model': model, 'coef': model.coef_,
            'intercept': model.intercept_,
            'rmse': rmse, 'mae': mae, 'r2': r2}

In [9]:
vals = {'region': [], 'RMSE': [], 'MAE': [], 'R2': []}

### Модель для региона 1

In [10]:
result_0 = train_and_evaluate(region_0)
vals['region'].append(0)
vals['RMSE'].append(result_0['rmse'])
vals['MAE'].append(result_0['mae'])
vals['R2'].append(result_0['r2'])
print(f"Коэффициенты модели [f1, f2, f3]:\n{result_0['coef']}\n\
Свободный коэффициент: {result_0['intercept']}")

Коэффициенты модели [f1, f2, f3]:
[  3.86230378 -14.28229978   6.58717561]
Свободный коэффициент: 77.66430534166085


### Модель для региона 2

In [11]:
result_1 = train_and_evaluate(region_1)
vals['region'].append(1)
vals['RMSE'].append(result_1['rmse'])
vals['MAE'].append(result_1['mae'])
vals['R2'].append(result_1['r2'])
print(f"Коэффициенты модели [f1, f2, f3]:\n{result_1['coef']}\n\
Свободный коэффициент: {result_1['intercept']}")

Коэффициенты модели [f1, f2, f3]:
[-1.44937615e-01 -2.17289070e-02  2.69524108e+01]
Свободный коэффициент: 1.65036975397922


### Модель для региона 3

In [12]:
result_2 = train_and_evaluate(region_2)
vals['region'].append(2)
vals['RMSE'].append(result_2['rmse'])
vals['MAE'].append(result_2['mae'])
vals['R2'].append(result_2['r2'])
print(f"Коэффициенты модели [f1, f2, f3]:\n{result_2['coef']}\n\
Свободный коэффициент: {result_2['intercept']}")

Коэффициенты модели [f1, f2, f3]:
[-0.08243043 -0.01898596  5.76713506]
Свободный коэффициент: 80.56850445282103


### Итоговые результаты

In [13]:
pd.DataFrame(vals)

,region,RMSE,MAE,R2
0,0,37.650638,30.960691,0.273962
1,1,0.890843,0.717607,0.999624
2,2,40.157688,32.840077,0.192751


## Подготовка к расчёту прибыли

In [14]:
budget = 1e10
income = 450000
n_samples = 500
top_k = 200

def profit(y_true, y_pred, n_samples=n_samples, top_k=top_k):
    indices = np.random.choice(len(y_true), size=n_samples, replace=False)
    sample_true = y_true.iloc[indices]
    sample_pred = y_pred[indices]

    top_indices = sample_pred.argsort()[-top_k:]
    selected_true = sample_true.iloc[top_indices]

    return selected_true.sum() * income - budget

def bootstrap(data, model, n_iter=1000):
    X = data[['f0', 'f1', 'f2']]
    y = data['product']
    y_pred = model.predict(X)

    profits = []
    for i in range(n_iter):
        profits.append(profit(y, y_pred))

    profits = np.array(profits)
    mean_profit = profits.mean()
    risk = (profits < 0).mean()

    return mean_profit, risk

## Расчёт прибыли и рисков

In [15]:
vals = {'region': [], 'mean_profit': [], 'risk': []}

mean_profit_0, risk_0 = bootstrap(region_0, result_0['model'])
vals['region'].append(0)
vals['mean_profit'].append(mean_profit_0)
vals['risk'].append(risk_0)

mean_profit_1, risk_1 = bootstrap(region_1, result_1['model'])
vals['region'].append(1)
vals['mean_profit'].append(mean_profit_1)
vals['risk'].append(risk_1)

mean_profit_2, risk_2 = bootstrap(region_2, result_2['model'])
vals['region'].append(2)
vals['mean_profit'].append(mean_profit_2)
vals['risk'].append(risk_2)

pd.DataFrame(vals)

,region,mean_profit,risk
0,0,4.181704e+08,0.054
1,1,4.532707e+08,0.017
2,2,3.657900e+08,0.092


Только регион 1 имеет вероятность убытков (риск) < 2.5% - выбираем его.  
Средняя прибыль 1 региона составляет 453.27 млн. рублей